<a href="https://colab.research.google.com/github/Shrideshi1/multi-label-email-risk-detection/blob/main/notebooks/01_data_preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = "/content/drive/MyDrive/Confidential_Risk_Project"
DATA_RAW_DIR = f"{PROJECT_DIR}/data/raw"
DATA_PROCESSED_DIR = f"{PROJECT_DIR}/data/processed"

%cd "$PROJECT_DIR"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Confidential_Risk_Project


In [ ]:
# Weeks 1–2: Data Collection, Cleaning, and Label Standardization
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import csv
import sys

from datasets import load_dataset

In [ ]:
# Increase the field size limit for the csv reader to handle large fields
csv.field_size_limit(sys.maxsize)

# Email datasets
messages_df = pd.read_csv(f"{DATA_RAW_DIR}/messages.csv", engine="python")
ceas_df = pd.read_csv(f"{DATA_RAW_DIR}/CEAS_08.csv", engine="python")

# Financial PhraseBank
financial_files = [
    f"{DATA_RAW_DIR}/Sentences_50Agree.txt",
    f"{DATA_RAW_DIR}/Sentences_66Agree.txt",
    f"{DATA_RAW_DIR}/Sentences_75Agree.txt",
    f"{DATA_RAW_DIR}/Sentences_AllAgree.txt"
]

financial_raw = pd.concat(
    [
        pd.read_csv(
            file,
            header=None,
            names=["raw"],
            encoding="latin-1",
            sep="\0",
            engine="python"
        )
        for file in financial_files
    ],
    ignore_index=True
)

# CUAD Master Clauses
master_df = pd.read_csv(f"{DATA_RAW_DIR}/master_clauses.csv")

In [ ]:
financial_df = financial_raw.copy()
financial_df[["text", "sentiment"]] = (financial_df["raw"].str.rsplit("@", n=1, expand=True))
financial_df = (financial_df.drop(columns="raw").drop_duplicates().reset_index(drop=True))

In [ ]:
#confirm adequate sample size.
datasets = {"Messages": messages_df.shape, "CEAS": ceas_df.shape, "Financial": financial_df.shape, "Master Clauses": master_df.shape}

for name, shape in datasets.items():print(f"{name}: {shape}")

Messages: (2893, 3)
CEAS: (39154, 7)
Financial: (4840, 2)
Master Clauses: (510, 83)


In [ ]:
financial_df["sentiment"].value_counts()

,count
sentiment,
neutral,2873
positive,1363
negative,604


The Financial PhraseBank contains three sentiment labels: neutral, positive, and negative. Neutral statements (2,873 samples) are the largest class and represent factual financial news or reports without a clear positive or negative opinion. Positive statements (1,363 samples) describe favorable financial outcomes or expectations, while negative statements (604 samples) indicate losses, risks, or unfavorable business events. The class imbalance suggests that neutral financial language dominates the dataset and should be considered during model development and evaluation.

In [ ]:
# Synthetic Confidential Dataset
# AI assistant (Claude) was utilized to assist with code development

import pandas as pd
import numpy as np

np.random.seed(780)

projects    = ["Atlas","Phoenix","Orion","Nova","Mercury","Sentinel","Falcon","Titan"]
departments = ["Finance","Engineering","Security","Legal","Executive Office","Research","Operations"]
dates       = ["Q3 2026","Q4 2026","January 2027","March 2027","June 2027"]
amounts     = [str(x) for x in range(25, 501, 5)]
tokens      = [str(x) for x in range(10_000_000, 99_999_999, 1_000_000)]
ids         = [str(x) for x in range(100_000, 999_999, 5_000)]
durations = [f"{n} months" for n in range(3, 37)] + [f"{n} weeks" for n in range(2, 26)] + [f"{n} quarters" for n in range(1, 9)]

# These variables were referenced in the fill function but not defined in the original notebook state.
# Initializing them to empty lists to prevent NameError, assuming they might be intended for future use
# or were placeholders not strictly required by the current templates.
counts = []
weekdays = []
timeframes = []

routing_phrases = [
    "Please route to {dept} only.",
    "For {dept} review prior to any distribution.",
    "Shared with {dept} pending final approval.",
    "Not for circulation beyond {dept} at this time.",
    "Awaiting sign-off from {dept} before wider release.",
    "Flagged for {dept} attention before next steps.",
    "Handle per {dept} protocol.",
    "Circulate within {dept} only until further notice.",
    "Requires {dept} acknowledgment before forwarding.",
    "Escalate to {dept} if questions arise.",
]

financial_templates = [
    "The {dept} team has completed the {date} revenue analysis for Project {project}. Total projected income is ${amount}M, with operating margins slightly below target.",
    "Attached is the budget breakdown for Project {project} as requested by {dept}. Figures reflect a ${amount}M spend plan through {date}.",
    "Following {timeframe}'s board meeting, the updated valuation for Project {project} has been revised to ${amount}M. Please do not forward until {dept} sign-off.",
    "Project {project} financials for {date}: gross revenue ${amount}M, net operating income pending audit review by {dept}.",
    "The {dept} office has finalized capital planning for Project {project}. Allocation stands at ${amount}M pending executive approval.",
    "Per the discussion with {dept}, Project {project} acquisition modeling puts fair value at ${amount}M. These figures are preliminary.",
    "Year-end projections for Project {project} prepared by {dept} indicate a ${amount}M variance from the original forecast.",
    "The {date} operating budget for Project {project} has been revised upward to ${amount}M following {dept} review.",
    "Interim financial disclosure for Project {project}: ${amount}M has been allocated through {date}. {dept} is coordinating next steps.",
    "Project {project} cost analysis submitted to {dept}. Current burn rate projects ${amount}M by {date}.",
    "Merger impact assessment for Project {project} estimates a ${amount}M adjustment. {dept} has requested this remain internal.",
    "Revised P&L for Project {project} reflects ${amount}M in deferred expenses. {dept} audit scheduled for {date}.",
]

credential_templates = [
    "The deployment pipeline for {dept} has been updated. New service key: SK{token}. Rotate after initial use.",
    "Reminder: the rsa-{token} key pair was issued for the Project {project} infrastructure. Do not commit to version control.",
    "VPN-{token} has been provisioned for the {dept} remote access cluster. Expires {date}.",
    "db-{token} is the updated admin credential for the Project {project} production database. Shared only with {dept} leads.",
    "oauth-{token} replaces the previous client secret for the {dept} authentication service. Update all environment configs.",
    "The svc-{token} account was created for automated deployments in Project {project}. Permissions are elevated — handle accordingly.",
    "cert-{token} has been issued for the {dept} environment. Valid through {date}; renewal is the responsibility of {dept}.",
    "Temporary access token tmp-{token} was generated for the Project {project} staging environment. Expires after {date}.",
    "Encryption key enc-{token} has been rotated for {dept} storage systems. Update dependent services before {date}.",
    "The api-{token} credential for Project {project} has been reissued following the {date} security review by {dept}.",
    "Root credential root-{token} for the {dept} cluster was reset. Distribution limited to {dept} infrastructure leads.",
    "Session signing key sess-{token} is now active for Project {project}. Previous key invalidated as of {date}.",
]

customer_templates = [
    "Account {id} has submitted a billing update request. The new payment method is on file with {dept}.",
    "Client {id} flagged a discrepancy in their invoice from {date}. {dept} is reviewing the transaction history.",
    "Vendor {id} has updated routing instructions for outgoing payments. Changes take effect {date}.",
    "Payroll adjustments for employee record {id} were processed by {dept} on {date}. Net changes reflect the new compensation structure.",
    "Partner {id} contract renewal includes revised payment terms. {dept} has the updated agreement.",
    "Billing profile {id} was flagged during the {date} audit. {dept} is coordinating with the client directly.",
    "Customer {id} requested a full account history export. {dept} is preparing the documentation as of {date}.",
    "Vendor {id} dispute has been escalated to {dept}. Original transaction dated {date}.",
    "Employee record {id} reflects updated direct deposit information effective {date}. Processed by {dept}.",
    "Client {id} data transfer request received {date}. {dept} is reviewing for compliance before fulfillment.",
    "Account {id} was placed under review following irregular activity detected by {dept} on {date}.",
    "Partner {id} submitted updated W-9 documentation to {dept}. Effective for payments beginning {date}.",
]

proprietary_templates = [
    "The {dept} team has documented the revised architecture for Project {project}. Key design decisions are outlined in the attached draft.",
    "Project {project} prototype testing concluded {date}. Performance results have been shared internally with {dept} only.",
    "The algorithm powering Project {project} was developed over {duration} by {dept}. Implementation notes are in the internal wiki.",
    "Source code for Project {project} has been moved to a private repository. Access is limited to {dept} engineers.",
    "Technical roadmap for Project {project} outlines feature development through {date}. Reviewed by {dept} leadership {timeframe}.",
    "Project {project} design specifications reflect {duration} of R&D by {dept}. The approach has not been published externally.",
    "Benchmarking results for Project {project} compiled by {dept} through {date}. Methodology is undocumented externally.",
    "The {dept} team has filed an internal patent disclosure for the Project {project} core mechanism after {duration} of development.",
    "Experimental feature set for Project {project} is under active development by {dept}. No external timeline has been communicated.",
    "Project {project} system integration notes prepared by {dept} following {date} testing cycle.",
    "Internal performance baseline for Project {project} established by {dept} over {duration}. Results deviate significantly from published benchmarks.",
    "Competitive analysis tied to Project {project} completed by {dept} after {duration} of market research. Contains forward-looking assumptions through {date}.",
]

legal_templates = [
    "The NDA between our organization and the Project {project} partner remains in effect through {date}. {dept} holds the signed copy.",
    "License terms for Project {project} were renegotiated {timeframe}. {dept} reviewed the updated language prior to execution.",
    "The amendment to the Project {project} contract introduces new liability clauses. Legal review by {dept} is pending.",
    "Export compliance documentation for Project {project} must be filed with {dept} before {date}.",
    "Settlement discussions for Project {project} are ongoing after {duration} of negotiations. {dept} has advised limiting written communications.",
    "IP assignment for Project {project} was executed {timeframe}. {dept} retains the original signed agreement.",
    "Indemnification clause revision for Project {project} is under review by {dept}. Finalization expected by {date}.",
    "Regulatory filing for Project {project} submitted by {dept} on {date}. Awaiting agency response.",
    "Arbitration proceedings related to Project {project} are scheduled for {date} after {duration} of dispute. {dept} is preparing documentation.",
    "Confidentiality obligations under the Project {project} agreement extend to all {dept} personnel through {date}.",
    "Force majeure provisions in the Project {project} contract were triggered {timeframe}. {dept} is assessing downstream impact.",
    "Intellectual property audit for Project {project} initiated by {dept} after {duration} review cycle. Preliminary findings expected by {date}.",
]

attachment_templates = [
    "Please find attached Budget_{project}.xlsx with the {date} figures. Prepared for {dept} review.",
    "Architecture_{project}.pdf has been updated to reflect the changes discussed with {dept} {timeframe}.",
    "Contract_{project}.docx contains the revised terms. {dept} requested this be distributed only to signing parties.",
    "SourceCode_{project}.zip includes the latest build after {duration} of development. Do not distribute outside {dept} without prior approval.",
    "Roadmap_{project}.pptx has been updated for the {date} leadership review. Prepared by {dept}.",
    "PayrollData_{project}.csv reflects {date} compensation changes. Routing to {dept} for processing.",
    "Audit_{project}.xlsx was generated by {dept} following {duration} of review activity.",
    "LegalDraft_{project}.docx is the working version as of {date}. {dept} has not yet signed off.",
    "IncidentReport_{project}.pdf was filed by {dept} on {date} after {duration} of investigation. Distribution pending closure.",
    "CompetitiveAnalysis_{project}.pptx prepared by {dept} after {duration} of market research for the {date} strategy session.",
    "AccessLog_{project}.csv exported by {dept} for the {date} security audit.",
    "DesignReview_{project}.pdf circulated to {dept} ahead of the {date} milestone after {duration} of iteration.",
]

def fill(text):
    filled = (text
        .replace("{project}",  np.random.choice(projects))
        .replace("{amount}",   np.random.choice(amounts))
        .replace("{token}",    np.random.choice(tokens))
        .replace("{id}",       np.random.choice(ids))
        .replace("{date}",     np.random.choice(dates))
        .replace("{duration}", np.random.choice(durations))
        .replace("{count}",    np.random.choice(counts) if counts else "")
        .replace("{weekday}",  np.random.choice(weekdays) if weekdays else "")
        .replace("{timeframe}",np.random.choice(timeframes) if timeframes else ""))
    while "{dept}" in filled:
        filled = filled.replace("{dept}", np.random.choice(departments), 1)
    return filled

def make_sample(template):
    routing = np.random.choice(routing_phrases)
    return fill(template) + " " + fill(routing)

template_map = {
    "financial":   financial_templates,
    "credentials": credential_templates,
    "customer":    customer_templates,
    "proprietary": proprietary_templates,
    "legal":       legal_templates,
    "attachment":  attachment_templates,
}

synthetic_df = pd.concat([
    pd.DataFrame({
        "text":     [make_sample(np.random.choice(template_map[cat])) for _ in range(250)],
        "category": cat,
        "source":   "synthetic",
    })
    for cat in template_map
], ignore_index=True)

print("Dataset shape:", synthetic_df.shape)
print("Unique texts: ", synthetic_df["text"].nunique(), "/", len(synthetic_df))
display(synthetic_df.sample(10, random_state=780))

Dataset shape: (1500, 3)
Unique texts:  1499 / 1500


,text,category,source
1493,LegalDraft_Atlas.docx is the working version a...,attachment,synthetic
425,Session signing key sess-69000000 is now activ...,credentials,synthetic
656,Client 595000 data transfer request received Q...,customer,synthetic
239,"Per the discussion with Security, Project Tita...",financial,synthetic
275,VPN-59000000 has been provisioned for the Lega...,credentials,synthetic
795,Project Titan prototype testing concluded Q4 2...,proprietary,synthetic
1362,LegalDraft_Mercury.docx is the working version...,attachment,synthetic
1016,Force majeure provisions in the Project Falcon...,legal,synthetic
630,Account 505000 was placed under review followi...,customer,synthetic
1442,Please find attached Budget_Sentinel.xlsx with...,attachment,synthetic


In [ ]:
#generate mixed synthetic samples
mixed_templates = [
    {
        "category": "financial_attachment",
        "labels": ["financial_risk", "attachment_risk"],
        "template": "Attached Budget_{project}.xlsx contains the {date} revenue forecast of ${amount}M. Please keep this within {dept}."
    },
    {
        "category": "credential_proprietary",
        "labels": ["credential_risk", "proprietary_risk"],
        "template": "Project {project} source code was moved to a private repository. Temporary access token api-{token} is limited to {dept}."
    },
    {
        "category": "customer_legal",
        "labels": ["customer_info_risk", "legal_risk"],
        "template": "Client {id} contract amendment includes updated liability terms. {dept} should review before {date}."
    },
    {
        "category": "financial_customer",
        "labels": ["financial_risk", "customer_info_risk"],
        "template": "Account {id} shows a billing adjustment of ${amount}M for {date}. {dept} is reviewing the transaction history."
    },
    {
        "category": "legal_attachment",
        "labels": ["legal_risk", "attachment_risk"],
        "template": "Attached Contract_{project}.docx contains revised indemnification language awaiting {dept} approval."
    },
    {
        "category": "credential_phishing",
        "labels": ["credential_risk", "phishing_spam_risk"],
        "template": "Urgent: verify your mailbox password using temporary token tmp-{token} before access expires on {date}."
    }
]

mixed_rows = []

for item in mixed_templates:
    for _ in range(750):
        mixed_rows.append({
            "text": fill(item["template"]) + " " + fill(np.random.choice(routing_phrases)),
            "category": item["category"],
            "source": "synthetic_mixed"
        })

mixed_df = pd.DataFrame(mixed_rows)

synthetic_df = pd.concat([synthetic_df, mixed_df], ignore_index=True)

In [ ]:
synthetic_df.to_csv(f"{DATA_PROCESSED_DIR}/synthetic_confidential.csv", index=False)
print("Saved to synthetic_confidential.csv")

Saved to synthetic_confidential.csv


In [ ]:
# Combining all data sources in preparation for ML techniques
import pandas as pd

# Messages
messages_df = pd.read_csv(f"{DATA_RAW_DIR}/messages.csv", engine="python")[["subject", "message", "label"]]
messages_df["text"] = messages_df["subject"].fillna("") + " " + messages_df["message"].fillna("")
messages_df["category"] = messages_df["label"].map({0: "ham", 1: "spam"})
messages_df = messages_df[["text", "category"]].assign(source="messages")

# CEAS
ceas_df = pd.read_csv(f"{DATA_RAW_DIR}/CEAS_08.csv", engine="python")[["subject", "body", "label"]]
ceas_df["text"] = ceas_df["subject"].fillna("") + " " + ceas_df["body"].fillna("")
ceas_df["category"] = ceas_df["label"].map({0: "ham", 1: "phishing"})
ceas_df = ceas_df[["text", "category"]].assign(source="ceas")

# Financial PhraseBank
fin_rows = []

with open(f"{DATA_RAW_DIR}/Sentences_AllAgree.txt", encoding="latin-1") as f:
    for line in f:
        if "@" in line:
            text, category = line.strip().rsplit("@", 1)
            fin_rows.append({"text": text, "category": category})

financial_df = pd.DataFrame(fin_rows).assign(source="financial")

# CUAD
master_df = pd.read_csv(f"{DATA_RAW_DIR}/master_clauses.csv")
answer_cols = [c for c in master_df.columns if c.endswith("-Answer")]

cuad_rows = []

for _, row in master_df.iterrows():
    for col in answer_cols:
        if pd.notna(row[col]) and str(row[col]).strip():
            cuad_rows.append({
                "text": str(row[col]).strip(),
                "category": col.replace("-Answer", ""),
                "source": "cuad"
            })

cuad_df = pd.DataFrame(cuad_rows)

# Synthetic
synthetic_df = pd.read_csv(f"{DATA_PROCESSED_DIR}/synthetic_confidential.csv")
synthetic_df = synthetic_df[["text", "category", "source"]]

# Combine
combined_df = pd.concat(
    [messages_df, ceas_df, financial_df, cuad_df, synthetic_df],
    ignore_index=True
)

print(combined_df.shape)
print(combined_df["source"].value_counts())
display(combined_df.head())

(69910, 3)
source
ceas               39154
cuad               19599
synthetic_mixed     4500
messages            2893
financial           2264
synthetic           1500
Name: count, dtype: int64


,text,category,source
0,job posting - apple-iss research center conten...,ham,messages
1,"lang classification grimes , joseph e . and b...",ham,messages
2,query : letter frequencies for text identifica...,ham,messages
3,risk a colleague and i are researching the dif...,ham,messages
4,request book information earlier this morning ...,ham,messages


The combined dataset provides a diverse and realistic collection of organizational communications by integrating general email messages, phishing and spam emails, financial text, legal contract clauses, and synthetic confidential examples into a single data source. Rather than artificially balancing the classes, the dataset preserves the natural prevalence of benign (ham) and phishing communications, reflecting real-world enterprise environments where routine emails significantly outnumber sensitive or malicious content. This design enables the model to learn normal communication patterns while simultaneously identifying phishing attempts and potential confidential-information leakage, resulting in a practical benchmark for evaluating secure email and data loss prevention systems.




In [ ]:
# Multi-label risk mapping for project labels

risk_columns = [
    "financial_risk",
    "credential_risk",
    "customer_info_risk",
    "proprietary_risk",
    "legal_risk",
    "attachment_risk",
    "phishing_spam_risk"
]


# Initialize all risk labels to 0
for col in risk_columns:
    combined_df[col] = 0

# Optional single-label summary column for reporting only
summary_mapping = {
    "ham": "Ham",
    "spam": "Phishing/Spam",
    "phishing": "Phishing/Spam",
    "positive": "Financial",
    "neutral": "Financial",
    "negative": "Financial",
    "financial": "Financial",
    "credentials": "Confidential",
    "customer": "Confidential",
    "proprietary": "Confidential",
    "attachment": "Confidential",
    "legal": "Legal",
    "financial_attachment": "Confidential",
    "credential_proprietary": "Confidential",
    "customer_legal": "Confidential",
    "financial_customer": "Confidential",
    "legal_attachment": "Confidential",
    "credential_phishing": "Confidential"
}

combined_df["project_category"] = (
    combined_df["category"]
    .map(summary_mapping)
    .fillna("Legal")
)

# Single-risk labels
combined_df.loc[
    combined_df["category"].isin(["positive", "neutral", "negative", "financial"]),
    "financial_risk"
] = 1

combined_df.loc[
    combined_df["category"].eq("credentials"),
    "credential_risk"
] = 1

combined_df.loc[
    combined_df["category"].eq("customer"),
    "customer_info_risk"
] = 1

combined_df.loc[
    combined_df["category"].eq("proprietary"),
    "proprietary_risk"
] = 1

combined_df.loc[
    combined_df["category"].eq("attachment"),
    "attachment_risk"
] = 1

combined_df.loc[
    (combined_df["category"].eq("legal")) | (combined_df["source"].eq("cuad")),
    "legal_risk"
] = 1

combined_df.loc[
    combined_df["category"].isin(["spam", "phishing"]),
    "phishing_spam_risk"
] = 1

# Mixed synthetic multi-label categories
mixed_label_map = {
    "financial_attachment": ["financial_risk", "attachment_risk"],
    "credential_proprietary": ["credential_risk", "proprietary_risk"],
    "customer_legal": ["customer_info_risk", "legal_risk"],
    "financial_customer": ["financial_risk", "customer_info_risk"],
    "legal_attachment": ["legal_risk", "attachment_risk"],
    "credential_phishing": ["credential_risk", "phishing_spam_risk"]
}

for category, labels in mixed_label_map.items():
    for label in labels:
        combined_df.loc[combined_df["category"].eq(category), label] = 1

# Final risk score = number of active risk labels
combined_df["risk_score"] = combined_df[risk_columns].sum(axis=1)

X = combined_df["text"]
y = combined_df[risk_columns]


# Checks
print("Project category counts:")
display(combined_df["project_category"].value_counts())

print("Multi-label risk counts:")
display(
    combined_df[risk_columns]
    .sum()
    .sort_values(ascending=False)
    .rename_axis("Risk Label")
    .reset_index(name="Count")
)

print("Number of risk labels per row:")
display(
    combined_df[risk_columns]
    .sum(axis=1)
    .value_counts()
    .sort_index()
    .rename_axis("Number of Risk Labels")
    .reset_index(name="Rows")
)

Project category counts:


,count
project_category,
Phishing/Spam,22323
Legal,19849
Ham,19724
Confidential,5500
Financial,2514


Multi-label risk counts:


,Risk Label,Count
0,phishing_spam_risk,23073
1,legal_risk,21349
2,financial_risk,4014
3,customer_info_risk,1750
4,credential_risk,1750
5,attachment_risk,1750
6,proprietary_risk,1000


Number of risk labels per row:


,Number of Risk Labels,Rows
0,0,19724
1,1,45686
2,2,4500


In [ ]:
print("Dataset shape:", combined_df.shape)

display(combined_df.head())

print("Single-label project category counts:")
display(combined_df["project_category"].value_counts())

risk_columns = [
    "financial_risk",
    "credential_risk",
    "customer_info_risk",
    "proprietary_risk",
    "legal_risk",
    "attachment_risk",
    "phishing_spam_risk"
]

print("Multi-label risk counts:")
display(
    combined_df[risk_columns]
    .sum()
    .sort_values(ascending=False)
    .rename_axis("Risk Label")
    .reset_index(name="Count")
)

print("Number of risk labels per row:")
display(
    combined_df[risk_columns]
    .sum(axis=1)
    .value_counts()
    .sort_index()
    .rename_axis("Number of Risk Labels")
    .reset_index(name="Rows")
)

print("Missing values:")
print(combined_df.isnull().sum())

Dataset shape: (69910, 12)


,text,category,source,financial_risk,credential_risk,customer_info_risk,proprietary_risk,legal_risk,attachment_risk,phishing_spam_risk,project_category,risk_score
0,job posting - apple-iss research center conten...,ham,messages,0,0,0,0,0,0,0,Ham,0
1,"lang classification grimes , joseph e . and b...",ham,messages,0,0,0,0,0,0,0,Ham,0
2,query : letter frequencies for text identifica...,ham,messages,0,0,0,0,0,0,0,Ham,0
3,risk a colleague and i are researching the dif...,ham,messages,0,0,0,0,0,0,0,Ham,0
4,request book information earlier this morning ...,ham,messages,0,0,0,0,0,0,0,Ham,0


Single-label project category counts:


,count
project_category,
Phishing/Spam,22323
Legal,19849
Ham,19724
Confidential,5500
Financial,2514


Multi-label risk counts:


,Risk Label,Count
0,phishing_spam_risk,23073
1,legal_risk,21349
2,financial_risk,4014
3,customer_info_risk,1750
4,credential_risk,1750
5,attachment_risk,1750
6,proprietary_risk,1000


Number of risk labels per row:


,Number of Risk Labels,Rows
0,0,19724
1,1,45686
2,2,4500


Missing values:
text                  0
category              0
source                0
financial_risk        0
credential_risk       0
customer_info_risk    0
proprietary_risk      0
legal_risk            0
attachment_risk       0
phishing_spam_risk    0
project_category      0
risk_score            0
dtype: int64


In [ ]:
combined_df.to_csv(f"{DATA_PROCESSED_DIR}/combined_dataset.csv", index=False)
print("Saved to combined_dataset.csv")

Saved to combined_dataset.csv


The combined dataset contains five primary communication categories: ham, phishing/spam, financial, legal, and confidential. Ham, phishing/spam, and legal messages make up the largest portions of the dataset, reflecting the volume of routine organizational communication, malicious email examples, and contract-based legal language available from public datasets. Financial text provides examples of specialized business language, while the confidential category represents multiple forms of sensitive organizational information, including credentials, customer/client records, proprietary project details, legal content, financial disclosures, and sensitive attachments. Unlike a traditional single-label spam detection dataset, this dataset also includes multi-label confidential-risk examples in which one communication may contain more than one type of risk, such as financial information with an attachment or credentials connected to proprietary project details. This unified dataset provides a realistic benchmark for developing and evaluating a machine learning framework capable of distinguishing routine communications from phishing attempts while identifying multiple confidential-information leakage risks.
The confidential category contains 5,500 curated synthetic examples representing both single-risk and mixed-risk organizational communications. While smaller than the ham, phishing/spam, and legal categories, this distribution reflects real-world enterprise settings where confidential communications are less frequent but potentially high impact. The final dataset contains 69,910 text samples, including 19,724 samples with no risk labels, 45,686 samples with one risk label, and 4,500 samples with two simultaneous risk labels. This structure directly supports the project’s multi-label objective because some communications may pose multiple risks at the same time.

During Weeks 1–2, we collected and explored multiple datasets representing different forms of organizational communication, including email messages, phishing emails, financial statements, and legal contract language. Initial data exploration consisted of previewing each dataset, examining dataset dimensions, reviewing category distributions, and checking for missing values to verify data quality before preprocessing. The Financial PhraseBank dataset was further analyzed through its sentiment distribution, showing that neutral statements comprise the majority of financial communications, followed by positive and negative statements. These exploratory steps provided a better understanding of the structure, content, and potential challenges associated with each data source.

To address the limited availability of publicly available confidential organizational communications, we developed a synthetic confidential-content dataset representing financial information, credentials, customer/client records, proprietary project details, legal language, and sensitive attachments. The dataset was generated using contextual templates with randomized projects, departments, identifiers, dates, monetary values, access tokens, and document types to create realistic organizational text rather than simple keyword examples. We also added mixed-risk synthetic examples, such as financial attachments, credential-related proprietary content, customer-related legal content, and credential phishing messages, so the dataset supports true multi-label classification.


The resulting combined dataset contains 69,910 samples with structured ground-truth labels across seven risk categories: financial risk, credential risk, customer information risk, proprietary risk, legal risk, attachment risk, and phishing/spam risk. Of these samples, 4,500 contain two simultaneous risk labels, allowing the project to model cases where one communication creates multiple information-leakage concerns. This synthetic data serves as a safe research proxy for sensitive organizational communications and simulates potential information-leakage scenarios without exposing actual confidential data. In a production setting, the synthetic data would ideally be replaced or supplemented with appropriately authorized and anonymized internal communications to better capture real-world language and document patterns while maintaining privacy. Together, the public and synthetic datasets establish a comprehensive foundation for feature engineering and comparison of TF-IDF, semantic embeddings, metadata, and machine learning models for multi-label confidential-content risk detection.
